In [1]:
import openai,json
client = openai.OpenAI()
messages = []


In [3]:
import requests

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    response.raise_for_status()
    return response.json()[:10]

def get_movie_id_by_name(name):
    movies = get_popular_movies()
    normalized_name = name.strip().lower()

    exact_matches = [
        movie for movie in movies
        if movie.get("title", "").strip().lower() == normalized_name
        or movie.get("original_title", "").strip().lower() == normalized_name
    ]

    partial_matches = [
        movie for movie in movies
        if normalized_name in movie.get("title", "").strip().lower()
        or normalized_name in movie.get("original_title", "").strip().lower()
    ]

    matches = exact_matches or partial_matches

    return [
        {
            "id": movie["id"],
            "title": movie.get("title"),
            "original_title": movie.get("original_title"),
            "release_date": movie.get("release_date")
        }
        for movie in matches[:5]
    ]

def resolve_movie_id(id=None, name=None):
    if id is not None:
        return id
    if not name:
        raise ValueError("Either id or name must be provided.")

    matches = get_movie_id_by_name(name)
    if not matches:
        raise ValueError(f"No movie found for '{name}'.")

    return matches[0]["id"]

def get_movie_details(id=None, name=None):
    movie_id = resolve_movie_id(id=id, name=name)
    response = requests.get(f"{BASE_URL}/movies/{movie_id}")
    response.raise_for_status()
    return response.json()

def get_similar_movies(id=None, name=None):
    movie_id = resolve_movie_id(id=id, name=name)
    response = requests.get(f"{BASE_URL}/movies/{movie_id}/similar")
    response.raise_for_status()
    return response.json()

FUNCTION_MAP = {
    'get_popular_movies': get_popular_movies,
    'get_movie_id_by_name': get_movie_id_by_name,
    'get_movie_details': get_movie_details,
    'get_similar_movies':get_similar_movies
}

In [4]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get the list of popular movies",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_id_by_name",
            "description": "Find movie IDs by movie title using the current popular movies list",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "The movie title to search for",
                    }
                },
                "required": ["name"]
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information for a specific movie by id or movie title",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The ID of the movie",
                    },
                    "name": {
                        "type": "string",
                        "description": "The movie title to search for when id is not known",
                    }
                },
                "required": []
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get similar movies for a specific movie by id or movie title",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The ID of the movie",
                    },
                    "name": {
                        "type": "string",
                        "description": "The movie title to search for when id is not known",
                    }
                },
                "required": []
            },
        },
    }
]

In [5]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message : ChatCompletionMessage):
    if message.tool_calls:
        messages.append({
            "role": "assistant",
            "content":message.content or"",
            "tool_calls": [{
                "id":tool_call.id,
                "type":"function",
                "function":{
                    "name" : tool_call.function.name,
                    "arguments": tool_call.function.arguments
                }
            } for tool_call in message.tool_calls]
        })

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function : {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments={}
            
            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append({
                "role": "tool",
                "tool_call_id":tool_call.id,
                "name":function_name,
                "content": json.dumps(result, ensure_ascii=False),

            })


        call_ai()
    else:
        messages.append({"role": "assistant","content": message.content})
        print(f"AI: {message.content}")

In [6]:
def call_ai():
    response = client.chat.completions.create(
    model= "gpt-4o-mini",
    messages=messages,
    tools=TOOLS
)
    process_ai_response(response.choices[0].message)
   

In [8]:
while True:
    message = input("LLM에 보낼 메세지를 입력하세요")
    if message == "quit" or message == "q":
        break
    else:
        messages.append(
            {
                "role":"user",
                "content": message
            }
        )
        print(f"User: {message}")
        call_ai()






User: 듄에 대해 알려줘
Calling function : get_movie_id_by_name with {"name":"Dune"}
Ran get_movie_id_by_name with args {'name': 'Dune'} for a result of []
Calling function : get_movie_details with {"name":"Dune"}


ValueError: No movie found for 'Dune'.

In [ ]:
messages